In [ ]:
# scheduler.py
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple


# ============================================================
# Request definition
# ============================================================

@dataclass
class Request:
    req_id: int
    arrival_time: int
    prompt_len: int
    output_len: int
    generated: int = 0
    start_time: int | None = None
    finish_time: int | None = None

    def is_finished(self) -> bool:
        return self.generated >= self.output_len

    @property
    def total_len(self) -> int:
        return self.prompt_len + self.output_len


def generate_requests(
    num_requests: int = 512,
    arrival_gap_range: Tuple[int, int] = (0, 2),
    prompt_len_range: Tuple[int, int] = (64, 2048),
    output_len_range: Tuple[int, int] = (16, 256),
    seed: int = 0,
) -> List[Request]:
    random.seed(seed)

    requests = []
    t = 0

    for i in range(num_requests):
        t += random.randint(*arrival_gap_range)
        prompt_len = random.randint(*prompt_len_range)
        output_len = random.randint(*output_len_range)

        requests.append(
            Request(
                req_id=i,
                arrival_time=t,
                prompt_len=prompt_len,
                output_len=output_len,
            )
        )

    return requests


# ============================================================
# KV cache allocators
# ============================================================

class ContiguousKVAllocator:
    """
    Naive contiguous KV allocator.

    Each request reserves max_seq_len KV slots, regardless of its actual length.
    This simulates the memory waste problem in naive KV cache management.
    """

    def __init__(self, max_num_reqs: int, max_seq_len: int):
        self.max_num_reqs = max_num_reqs
        self.max_seq_len = max_seq_len
        self.active_reqs: Dict[int, Dict] = {}

    def allocate(self, req_id: int, actual_len: int) -> bool:
        if len(self.active_reqs) >= self.max_num_reqs:
            return False

        if actual_len > self.max_seq_len:
            return False

        self.active_reqs[req_id] = {
            "actual_len": actual_len,
            "allocated_tokens": self.max_seq_len,
        }
        return True

    def free(self, req_id: int):
        self.active_reqs.pop(req_id, None)

    def allocated_tokens(self) -> int:
        return sum(x["allocated_tokens"] for x in self.active_reqs.values())

    def used_tokens(self) -> int:
        return sum(x["actual_len"] for x in self.active_reqs.values())

    def waste_ratio(self) -> float:
        allocated = self.allocated_tokens()
        used = self.used_tokens()

        if allocated == 0:
            return 0.0

        return 1.0 - used / allocated

    def active_count(self) -> int:
        return len(self.active_reqs)


class PagedKVAllocator:
    """
    Paged KV allocator.

    Each request only reserves ceil(actual_len / block_size) physical blocks.
    This simulates the core memory-management idea of PagedAttention.
    """

    def __init__(
        self,
        num_physical_blocks: int,
        block_size: int,
        max_blocks_per_seq: int,
    ):
        self.num_physical_blocks = num_physical_blocks
        self.block_size = block_size
        self.max_blocks_per_seq = max_blocks_per_seq

        self.free_blocks = list(range(num_physical_blocks))
        self.req_to_blocks: Dict[int, List[int]] = {}
        self.req_to_actual_len: Dict[int, int] = {}

    def allocate(self, req_id: int, actual_len: int) -> bool:
        required_blocks = math.ceil(actual_len / self.block_size)

        if required_blocks > self.max_blocks_per_seq:
            return False

        if len(self.free_blocks) < required_blocks:
            return False

        blocks = []
        for _ in range(required_blocks):
            blocks.append(self.free_blocks.pop())

        self.req_to_blocks[req_id] = blocks
        self.req_to_actual_len[req_id] = actual_len

        return True

    def free(self, req_id: int):
        blocks = self.req_to_blocks.pop(req_id, [])
        self.req_to_actual_len.pop(req_id, None)
        self.free_blocks.extend(blocks)

    def allocated_tokens(self) -> int:
        used_blocks = self.num_physical_blocks - len(self.free_blocks)
        return used_blocks * self.block_size

    def used_tokens(self) -> int:
        return sum(self.req_to_actual_len.values())

    def waste_ratio(self) -> float:
        allocated = self.allocated_tokens()
        used = self.used_tokens()

        if allocated == 0:
            return 0.0

        return 1.0 - used / allocated

    def active_count(self) -> int:
        return len(self.req_to_blocks)

    def build_block_table(self, active_req_ids: List[int]):
        """
        Return a CPU-side block table.

        block_table[i][j] = physical block id
        where i is batch index and j is logical block index.

        In a real Triton kernel, this block table would be a CUDA tensor.
        """

        table = []

        for req_id in active_req_ids:
            row = [-1 for _ in range(self.max_blocks_per_seq)]
            blocks = self.req_to_blocks[req_id]

            for j, physical_block_id in enumerate(blocks):
                row[j] = physical_block_id

            table.append(row)

        return table


# ============================================================
# Static batching
# ============================================================

def run_static_batching(
    requests: List[Request],
    batch_size: int,
) -> Tuple[List[Request], List[int]]:
    """
    Static batching:

    1. Take a fixed batch.
    2. Decode this batch until all requests are finished.
    3. Only then take the next batch.

    This causes GPU under-utilization because short requests finish early,
    but no new requests are inserted into the batch.
    """

    time = 0
    completed = []
    waiting = list(requests)
    active_batch_sizes = []

    while waiting:
        batch = waiting[:batch_size]
        waiting = waiting[batch_size:]

        # Batch cannot start before all requests in this batch have arrived.
        time = max(time, max(r.arrival_time for r in batch))

        for r in batch:
            r.start_time = time

        while not all(r.is_finished() for r in batch):
            active = [r for r in batch if not r.is_finished()]
            active_batch_sizes.append(len(active))

            # Decode one token for each active request.
            for r in active:
                r.generated += 1

            time += 1

        for r in batch:
            r.finish_time = time
            completed.append(r)

    return completed, active_batch_sizes


# ============================================================
# Continuous batching
# ============================================================

def run_continuous_batching(
    requests: List[Request],
    batch_size: int,
) -> Tuple[List[Request], List[int]]:
    """
    Continuous batching:

    At every decoding step:
      1. Finished requests are removed.
      2. Newly arrived requests are inserted if capacity is available.
      3. Decode one token for each active request.

    This keeps active batch size high and improves throughput.
    """

    time = 0
    completed = []
    waiting = list(requests)
    active: List[Request] = []

    active_batch_sizes = []

    while waiting or active:
        # Remove finished requests.
        still_active = []
        for r in active:
            if r.is_finished():
                r.finish_time = time
                completed.append(r)
            else:
                still_active.append(r)

        active = still_active

        # Add newly arrived requests while there is capacity.
        while len(active) < batch_size and waiting and waiting[0].arrival_time <= time:
            r = waiting.pop(0)
            r.start_time = time
            active.append(r)

        # If there is no active request, jump to next arrival time.
        if not active and waiting:
            time = max(time, waiting[0].arrival_time)
            continue

        active_batch_sizes.append(len(active))

        # Decode one token for each active request.
        for r in active:
            r.generated += 1

        time += 1

    return completed, active_batch_sizes


# ============================================================
# Metrics
# ============================================================

def summarize_results(
    completed: List[Request],
    active_batch_sizes: List[int],
):
    latencies = [r.finish_time - r.arrival_time for r in completed]
    total_output_tokens = sum(r.output_len for r in completed)

    start_time = min(r.arrival_time for r in completed)
    finish_time = max(r.finish_time for r in completed)
    total_time = finish_time - start_time

    avg_latency = sum(latencies) / len(latencies)
    p95_latency = sorted(latencies)[int(0.95 * len(latencies))]

    throughput = total_output_tokens / total_time
    avg_active_batch = sum(active_batch_sizes) / len(active_batch_sizes)

    return {
        "total_output_tokens": total_output_tokens,
        "total_time": total_time,
        "throughput_tok_per_step": throughput,
        "avg_latency": avg_latency,
        "p95_latency": p95_latency,
        "avg_active_batch": avg_active_batch,
    }

In [ ]:
# benchmark.py
import os
import copy
import pandas as pd

from scheduler import (
    ContiguousKVAllocator,
    PagedKVAllocator,
    generate_requests,
    run_static_batching,
    run_continuous_batching,
    summarize_results,
)


def benchmark_kv_allocator():
    """
    Compare contiguous KV cache and paged KV cache.

    The two allocators use the same total KV token budget.

    Contiguous:
      each request reserves max_seq_len tokens.

    Paged:
      each request reserves ceil(actual_len / block_size) blocks.
    """

    os.makedirs("results", exist_ok=True)

    max_seq_len = 2048
    block_size = 16
    max_blocks_per_seq = max_seq_len // block_size

    max_num_reqs = 64
    total_kv_token_budget = max_seq_len * max_num_reqs

    contiguous = ContiguousKVAllocator(
        max_num_reqs=max_num_reqs,
        max_seq_len=max_seq_len,
    )

    paged = PagedKVAllocator(
        num_physical_blocks=total_kv_token_budget // block_size,
        block_size=block_size,
        max_blocks_per_seq=max_blocks_per_seq,
    )

    requests = generate_requests(
        num_requests=512,
        arrival_gap_range=(0, 2),
        prompt_len_range=(64, 2048),
        output_len_range=(16, 256),
        seed=42,
    )

    rows = []

    for allocator_name, allocator in [
        ("contiguous", contiguous),
        ("paged", paged),
    ]:
        accepted = 0
        rejected = 0

        for r in requests:
            actual_len = min(r.prompt_len + r.output_len, max_seq_len)

            ok = allocator.allocate(r.req_id, actual_len)

            if ok:
                accepted += 1
            else:
                rejected += 1

            rows.append(
                {
                    "allocator": allocator_name,
                    "step": accepted + rejected,
                    "accepted_reqs": accepted,
                    "rejected_reqs": rejected,
                    "active_reqs": allocator.active_count(),
                    "allocated_tokens": allocator.allocated_tokens(),
                    "used_tokens": allocator.used_tokens(),
                    "waste_ratio": allocator.waste_ratio(),
                }
            )

    df = pd.DataFrame(rows)
    df.to_csv("results/paged_vs_contiguous.csv", index=False)

    print("\n=== KV Allocator Comparison ===")
    print(df.groupby("allocator").tail(1).to_string(index=False))


def benchmark_batching():
    """
    Compare static batching and continuous batching.

    This is a scheduling-level simulation.

    Static batching:
      fixed batch, wait until all requests finish.

    Continuous batching:
      remove finished requests and insert new requests every decode step.
    """

    os.makedirs("results", exist_ok=True)

    batch_size = 32

    requests = generate_requests(
        num_requests=512,
        arrival_gap_range=(0, 2),
        prompt_len_range=(64, 1024),
        output_len_range=(16, 256),
        seed=123,
    )

    static_completed, static_active = run_static_batching(
        copy.deepcopy(requests),
        batch_size=batch_size,
    )

    continuous_completed, continuous_active = run_continuous_batching(
        copy.deepcopy(requests),
        batch_size=batch_size,
    )

    static_summary = summarize_results(static_completed, static_active)
    continuous_summary = summarize_results(continuous_completed, continuous_active)

    rows = [
        {"scheduler": "static", **static_summary},
        {"scheduler": "continuous", **continuous_summary},
    ]

    df = pd.DataFrame(rows)
    df.to_csv("results/static_vs_continuous.csv", index=False)

    print("\n=== Batching Policy Comparison ===")
    print(df.to_string(index=False))

    # Also save active batch size trace.
    active_df = pd.DataFrame(
        {
            "static_active_batch": pd.Series(static_active),
            "continuous_active_batch": pd.Series(continuous_active),
        }
    )
    active_df.to_csv("results/active_batch_trace.csv", index=False)


if __name__ == "__main__":
    benchmark_kv_allocator()
    benchmark_batching()

In [ ]:
# plot_results.py
import os
import pandas as pd
import matplotlib.pyplot as plt


def plot_results():
    os.makedirs("results", exist_ok=True)

    kv = pd.read_csv("results/paged_vs_contiguous.csv")
    sched = pd.read_csv("results/static_vs_continuous.csv")
    active_trace = pd.read_csv("results/active_batch_trace.csv")

    kv_last = kv.groupby("allocator").tail(1)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # ========================================================
    # Plot 1: KV waste ratio
    # ========================================================
    axes[0, 0].bar(
        kv_last["allocator"],
        kv_last["waste_ratio"],
    )
    axes[0, 0].set_title("Paged KV reduces memory waste")
    axes[0, 0].set_ylabel("KV waste ratio")
    axes[0, 0].set_xlabel("KV cache allocator")
    axes[0, 0].set_ylim(0, 1.0)

    # ========================================================
    # Plot 2: accepted requests under same KV budget
    # ========================================================
    axes[0, 1].bar(
        kv_last["allocator"],
        kv_last["accepted_reqs"],
    )
    axes[0, 1].set_title("Paged KV accepts more requests")
    axes[0, 1].set_ylabel("Accepted requests")
    axes[0, 1].set_xlabel("KV cache allocator")

    # ========================================================
    # Plot 3: throughput
    # ========================================================
    axes[1, 0].bar(
        sched["scheduler"],
        sched["throughput_tok_per_step"],
    )
    axes[1, 0].set_title("Continuous batching improves throughput")
    axes[1, 0].set_ylabel("Output tokens / step")
    axes[1, 0].set_xlabel("Batching policy")

    # ========================================================
    # Plot 4: active batch size trace
    # ========================================================
    axes[1, 1].plot(
        active_trace["static_active_batch"].dropna().values,
        label="static",
    )
    axes[1, 1].plot(
        active_trace["continuous_active_batch"].dropna().values,
        label="continuous",
    )
    axes[1, 1].set_title("Continuous batching keeps batch size high")
    axes[1, 1].set_ylabel("Active batch size")
    axes[1, 1].set_xlabel("Decode step")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig("results/vllm_micro_results.png", dpi=200)
    plt.show()


if __name__ == "__main__":
    plot_results()